# ch07 微调范式（练习版）

> 补全所有 `____`，跑通每个 `assert`。真值来自实跑，不是你算的。

**任务清单**

1. WordPiece 词表构建与 id 空间
2. 手写 `wp_encode`（贪心最长匹配 + `##` 续接）
3. BERT 的三个输入张量
4. MiniBERT 参数量手算对账
5. MLM 掩码策略（80/10/10）
6. 两档语料的 MLM 预训练
7. few-shot：预训练规模 × 下游数据量
8. `[CLS]` vs mean 池化
9. 手写 `LoRALinear`
10. LoRA 参数量与三策略对比
11. HF BERT 参数量恒等式
12. 把手写 LoRA 挂到 HF 模型上
13. 四种策略在 HF BERT 上真跑
14. 公平对照：随机初始化 vs 预训练权重
15. 用 Trainer 真跑（含默认 lr 调度的坑）

In [ ]:
import copy
import importlib.util
import math
import re
import time
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_num_threads(4)
print("torch", torch.__version__, "| 线程", torch.get_num_threads())

# §7.12 起用真实库；**仍然只建 config，不下载任何预训练权重**
import transformers
from transformers import (BertConfig, BertForSequenceClassification,
                          Trainer, TrainingArguments)

DEV_HF = "mps" if torch.backends.mps.is_available() else "cpu"
print("transformers", transformers.__version__, "| HF 训练设备", DEV_HF)

In [ ]:
from pathlib import Path

DATA = Path("data")                  # notebook cwd = coding/04_nlp
CSV = DATA / "synth_text.csv"
MAKE_DATA = DATA / "make_data.py"

df = pd.read_csv(CSV, keep_default_na=False)
print("读入", df.shape, "| 标签分布", df["缺陷类型"].value_counts().to_dict())
print("无效文本", int(df["缺陷描述"].str.strip().isin(["", "N/A", "-", "无"]).sum()), "条")

In [ ]:
# 脚手架：常量与工具函数两版都有，你只填 @@todo 块里的内容
DEVICES = ["变压器", "断路器", "隔离开关", "避雷器", "电流互感器", "绝缘子", "套管", "母线"]
SIGWORDS = {
    "渗漏油": ["渗油", "漏油", "油迹", "油污", "滴油", "油位下降"],
    "锈蚀": ["锈蚀", "生锈", "铁锈", "漆面剥落", "镀锌层脱落"],
    "破损": ["破损", "裂纹", "碎裂", "缺口", "掉瓷", "箱体变形"],
    "发热": ["发热", "温度偏高", "过热", "烫手", "温升异常"],
    "异物": ["异物", "鸟巢", "塑料袋", "树枝", "风筝线搭挂"],
}
LABELS5 = ["渗漏油", "锈蚀", "破损", "发热", "异物"]
LID = {c: i for i, c in enumerate(LABELS5)}
VOCAB = DEVICES + [w for ws in SIGWORDS.values() for w in ws]
VOCAB_SET = set(VOCAB)
MAXW = max(len(w) for w in VOCAB)
SPECIALS = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]
HAN = re.compile(r"[\u4e00-\u9fff]")

# 模型超参：2 层 / d=64 / 4 头 —— CPU 上秒级，且小到能"背下来"，正好暴露预训练的真实收益
D, H, NL, FF, ML, NCLS = 64, 4, 2, 128, 24, 5


def clean(t):
    t = re.sub(r"<[^>]+>", "", t)
    t = re.sub(r"https?://\S+", "", t)
    t = re.sub(r"[\u3000\s]+", "", t)
    return t.replace(",", "，").strip()


def tok(t):
    """复用 ch01 的两级切分 + 正向最大匹配"""
    out = []
    for seg in re.findall(r"[A-Za-z0-9_]+|[\u4e00-\u9fff]+|[^\sA-Za-z0-9_\u4e00-\u9fff]", t):
        if re.fullmatch(r"[\u4e00-\u9fff]+", seg):
            i = 0
            while i < len(seg):
                for L in range(min(MAXW, len(seg) - i), 0, -1):
                    if seg[i:i + L] in VOCAB_SET:
                        out.append(seg[i:i + L])
                        i += L
                        break
                else:
                    out.append(seg[i])
                    i += 1
        else:
            out.append(seg)
    return out


texts = [clean(t) for t in df["缺陷描述"]]
tokens = [tok(t) for t in texts]
bad = np.array([not t.strip() or t.strip() in {"N/A", "-", "无", "''", '""'} for t in texts])
kept = np.where(~bad)[0]
print("可用样本 %d | token 种类 %d | 总 token %d | 平均 token/句 %.6f"
      % (len(kept), len({w for i in kept for w in tokens[i]}),
         sum(len(tokens[i]) for i in kept), np.mean([len(tokens[i]) for i in kept])))

## 任务 1：WordPiece 建表与 id 空间

In [ ]:
# TODO(1)：WordPiece 建表：完整词 + 全部后缀 + 语料单字（单字也要有 "##" 形式）
# 提示：后缀是「续接片段」，一律加 "##" 前缀：{"##" + w[i:] for w in VOCAB for i in range(1, len(w))}
# 提示：语料里的汉字要连 "##单字" 一起进表，否则编码时无处可拆、整词退化 [UNK]
# 提示：最后 cont 取出所有 "##" 开头的片段，用来核对续接片段数量
wp_vocab = ____
cont = ____

# ---- 验收 ----
assert len(wp_vocab) == 316, f"词表应为 316，实际 {len(wp_vocab)}"
assert len(cont) == 151, f"续接片段应为 151，实际 {len(cont)}"
assert "##互感器" in wp_vocab and "电流互感器" in wp_vocab
print("WordPiece 词表 %d 词（续接片段 %d）| 例 %s" % (len(wp_vocab), len(cont), cont[:6]))

# 自建 id 映射：[PAD] [UNK] [CLS] [SEP] [MASK] 固定占前 5 位
ITEMS = SPECIALS + sorted(p for p in wp_vocab if p not in SPECIALS)
WID = {s: i for i, s in enumerate(ITEMS)}
NV = len(ITEMS)
PAD, UNK, CLS, SEP, MASK = (WID[t] for t in SPECIALS)
assert NV == 316 and (PAD, UNK, CLS, SEP, MASK) == (0, 1, 2, 3, 4)
print("id 空间 %d | [PAD]=%d [UNK]=%d [CLS]=%d [SEP]=%d [MASK]=%d" % (NV, PAD, UNK, CLS, SEP, MASK))

## 任务 2：WordPiece 编码

In [ ]:
# 函数骨架已给，你只需要填 @@todo 里的那一行
def wp_encode(word):
    """从左到右贪心取**最长**可匹配片段；续接片段带 "##" 前缀"""
    if word in wp_vocab:
        return [word]
    out, i = [], 0
    while i < len(word):
        L = len(word) - i
        while L > 0:
            # TODO(2)：构造候选片段：i==0 时是 word[i:i+L]；i>0 时必须是续接片段形式
            # 提示：续接片段 = "##" + word[i:i + L]；漏了 ## 就永远匹配不上
            piece = ____
            if piece in wp_vocab:
                out.append(piece)
                i += L
                break
            L -= 1
        else:
            return ["[UNK]"]          # 任一片段失败 -> 整词退化为 [UNK]（HF 行为）
    return out

# ---- 验收 ----
assert wp_encode("变压器") == ["变压器"], "整词命中就不拆"
assert wp_encode("请核实") == ["请", "##核", "##实"], "整词不在表 -> 首字 + ##续接"
assert wp_encode("严重") == ["严", "##重"]
assert wp_encode("铁芯") == ["[UNK]"], "有'铁'无'芯' -> 整词 [UNK]"
assert wp_encode("ZLX99") == ["[UNK]"], "字母串不在表"
for w in ["变压器", "电流互感器", "油位下降", "请核实", "严重", "铁芯"]:
    print("  %-10s -> %s" % (w, wp_encode(w)))
print()
print("与 BPE 的区别：BPE 编码时按**合并优先级**逐对合并（可能局部最优）；")
print("WordPiece 是**从头贪心取最长**，续接片段写 ## 前缀（词首不写）。")

## 任务 3：BERT 三个输入张量

In [ ]:
# 函数骨架已给，你只需要填 @@todo 里的内容
def bert_features(a, b=None, maxlen=ML):
    """把词序列拼成 BERT 的三个张量：input_ids / token_type_ids / attention_mask"""
    ta = [p for w in a for p in wp_encode(w)]
    # TODO(3)：单句：[CLS] + a 的片段 + [SEP]，seg 全 0
    # 提示：ids = [WID["[CLS]"]] + [WID.get(p, WID["[UNK]"]) for p in ta] + [WID["[SEP]"]]
    # 提示：seg 与 ids 等长，单句里每个位置都是 0
    ids = ____
    seg = ____
    if b is not None:
        tb = [p for w in b for p in wp_encode(w)]
        # TODO(4)：句对：接在已有的 [CLS] A [SEP] 后面，补 B 段与第二个 [SEP]
        # 提示：关键：第一个 [SEP] 的 seg 仍是 0（它属于 A 段），只有 B 段和末尾 [SEP] 是 1
        ids = ____
        seg = ____
    if len(ids) > maxlen:
        # TODO(5)：超长截断：最后一位要**强行写回 [SEP]**，seg 跟着取原末位
        # 提示：ids = ids[:maxlen-1] + [WID["[SEP]"]]；seg 同理保留 maxlen-1 位再补一位
        ids = ____
        seg = ____
    mask = [1] * len(ids)
    pad = maxlen - len(ids)
    return ids + [WID["[PAD]"]] * pad, seg + [0] * pad, mask + [0] * pad

# ---- 验收 ----
i1, s1, m1 = bert_features(["变压器", "渗油"])
i2, s2, m2 = bert_features(["变压器", "渗油"], ["设备", "缺陷"])
i3, s3, m3 = bert_features(["变压器", "渗油"], maxlen=3)
assert sum(m1) == 4 and i1[:4] == [WID["[CLS]"], WID["变压器"], WID["渗油"], WID["[SEP]"]]
assert set(s1[:4]) == {0} and m1[4] == 0, "单句 seg 全 0，pad 位 mask=0"
assert sum(m2) == 8 and s2[:8] == [0, 0, 0, 0, 1, 1, 1, 1], "句对 seg 前 0 后 1"
assert sum(m3) == 3 and i3 == [WID["[CLS]"], WID["变压器"], WID["[SEP]"]], "截断末位强制 [SEP]"
print("单句 ids[:5] =", i1[:5], "| seg[:5] =", s1[:5], "| mask[:5] =", m1[:5])
print("句对 seg[:8] =", s2[:8], "（前 4 个 0、后 4 个 1）")
print("截断 maxlen=3 ->", i3, "（末位被换成 [SEP]，保证句子闭合）")

## 任务 4：MiniBERT 与参数量

In [ ]:
class MHSA(nn.Module):
    """自写多头自注意力：q/k/v 三个独立 Linear —— 这样才能像 LoRA 论文那样只挂 Q/V"""

    def __init__(self, d=D, h=H):
        super().__init__()
        self.h, self.dh = h, d // h
        self.q, self.k, self.v, self.o = (nn.Linear(d, d) for _ in range(4))

    def forward(self, x, pad=None):
        B, T, _ = x.shape
        sh = (B, T, self.h, self.dh)
        q, k, v = (m(x).view(sh).transpose(1, 2) for m in (self.q, self.k, self.v))
        am = None if pad is None else (~pad)[:, None, None, :]
        o = F.scaled_dot_product_attention(q, k, v, attn_mask=am)
        return self.o(o.transpose(1, 2).reshape(B, T, -1))


class EncLayer(nn.Module):
    """Pre-LN 的 Encoder 层：ch05 已实测 Post-LN 在 6 层堆叠下首层梯度只剩末层的 0.000007"""

    def __init__(self, d=D, h=H, ff=FF):
        super().__init__()
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = MHSA(d, h)
        self.ff = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Linear(ff, d))

    def forward(self, x, pad=None):
        x = x + self.attn(self.n1(x), pad)
        return x + self.ff(self.n2(x))


class MiniBERT(nn.Module):
    """BERT 骨架：三种 embedding 相加 -> Encoder 堆叠 -> 池化 -> 分类头；另挂一个 MLM 头"""

    def __init__(self, nv=NV, d=D, h=H, nl=NL, maxlen=ML, ncls=NCLS, pool="cls"):
        super().__init__()
        self.tok = nn.Embedding(nv, d, padding_idx=0)
        self.seg = nn.Embedding(2, d)
        self.pos = nn.Embedding(maxlen, d)
        self.n0 = nn.LayerNorm(d)
        self.layers = nn.ModuleList([EncLayer(d, h, FF) for _ in range(nl)])
        self.pool = pool
        self.head = nn.Linear(d, ncls)
        self.mlm = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.LayerNorm(d), nn.Linear(d, nv))

    def body(self, ids, seg, mask):
        x = self.tok(ids) + self.seg(seg) + self.pos(torch.arange(ids.shape[1]))
        x = self.n0(x)
        pad = mask == 0
        for lyr in self.layers:
            x = lyr(x, pad)
        return x

    def forward(self, ids, seg, mask):
        hh = self.body(ids, seg, mask)
        if self.pool == "cls":
            pooled = hh[:, 0]
        else:
            m = mask[..., None].float()
            pooled = (hh * m).sum(1) / m.sum(1).clamp(min=1)
        return self.head(pooled), hh

    def mlm_logits(self, ids, seg, mask):
        return self.mlm(self.body(ids, seg, mask))


def mb(seed=0, **kw):
    with torch.random.fork_rng():
        torch.manual_seed(seed)
        return MiniBERT(**kw)


m0 = mb()
tot = sum(p.numel() for p in m0.parameters())
print("MiniBERT 参数量合计 %d" % tot)

In [ ]:
# TODO(6)：拆出四块参数量：嵌入（含前置 LayerNorm）/ Encoder / 分类头 / MLM 头
# 提示：嵌入 = tok.weight + seg.weight + pos.weight + n0 的 LayerNorm 参数
# 提示：enc_n = sum(p.numel() for p in m0.layers.parameters())，head / mlm 同理
emb_n = ____
enc_n = ____
head_n = ____
mlm_n = ____

# ---- 验收 ----
assert (NV, D, ML, FF) == (316, 64, 24, 128)
assert emb_n == 22016, f"嵌入应 22016，实际 {emb_n}"
assert enc_n == 66944, f"Encoder 应 66944，实际 {enc_n}"
assert head_n == 325 and mlm_n == 24828
assert tot == 114113, f"总量应 114113，实际 {tot}"
assert NV * D + 2 * D + ML * D + 2 * D == emb_n, "tok+seg+pos+LN = 22016"
assert 4 * D * D + 4 * D + 2 * D * FF + D + FF + 4 * D == enc_n // NL, "单层 Encoder 手算"
assert D * NCLS + NCLS == head_n
print("总参 %d = 嵌入 %d + Encoder %d + 分类头 %d + MLM头 %d" % (tot, emb_n, enc_n, head_n, mlm_n))
print("嵌入手算 tok %d + seg %d + pos %d + LayerNorm %d = %d"
      % (NV * D, 2 * D, ML * D, 2 * D, NV * D + 2 * D + ML * D + 2 * D))
print("单层 Encoder %d，手算 4d²+4d(attn) + 2·d·ff+d+ff(ff) + 4d(norm) = %d"
      % (enc_n // NL, 4 * D * D + 4 * D + 2 * D * FF + D + FF + 4 * D))
print("分类头 %d = %d×%d+%d，只占总量 %.4f%% —— 这是「微调很便宜」的一半原因"
      % (head_n, D, NCLS, NCLS, 100 * head_n / tot))

## 任务 5：MLM 掩码策略

In [ ]:
def featurize(seqs, maxlen=ML):
    I, S, M = [], [], []
    for tk in seqs:
        ids, seg, msk = bert_features(tk, maxlen=maxlen)
        I.append(ids)
        S.append(seg)
        M.append(msk)
    return torch.as_tensor(I), torch.as_tensor(S), torch.as_tensor(M)


I_all, S_all, M_all = featurize([tokens[i] for i in kept])
print("小语料张量 %s | 有效 token %d" % (tuple(I_all.shape), int(M_all.sum())))

In [ ]:
def mlm_batch(ids, mask, rng, p=0.15):
    """BERT 的掩码策略：15% 位置被选中，其中 80% [MASK] / 10% 随机词 / 10% 原样保留"""
    lab = ids.clone()
    cand = (mask == 1) & (ids != CLS) & (ids != SEP) & (ids != PAD)
    # TODO(7)：用第一个随机数决定「哪些位置被选中」，其余位置标签置 -100（不算损失）
    # 提示：sel = cand & (rng.random(ids.shape) < p)；lab[~sel] = -100
    sel = ____
    lab[~sel] = ____
    r = torch.as_tensor(rng.random(ids.shape), dtype=torch.float32)
    new = ids.clone()
    # TODO(8)：用第二个随机数把选中位置分成三份：80% 换 [MASK]、10% 换随机词、10% 不动
    # 提示：new[sel & (r < 0.8)] = MASK
    new[sel & (r < 0.8)] = ____
    new[sel & (r >= 0.8) & (r < 0.9)] = torch.as_tensor(
        rng.integers(1, NV, size=ids.shape), dtype=torch.long)[sel & (r >= 0.8) & (r < 0.9)]
    return new, lab

In [ ]:
ids_p, lab_p = mlm_batch(I_all, M_all, np.random.default_rng(1))
n_sel = int((lab_p != -100).sum())
n_mask = int((ids_p == MASK).sum())
n_rand = int(((ids_p != I_all) & (ids_p != MASK) & (lab_p != -100)).sum())
n_keep = int(((ids_p == I_all) & (lab_p != -100)).sum())

# ---- 验收 ----
assert n_sel == 1307, f"应选中 1307 个位置，实际 {n_sel}"
assert abs(n_sel / int(M_all.sum()) - 0.134798) < 1e-6, "占有效 token 的 13.48%"
assert n_mask == 1070, f"[MASK] 替换应 1070，实际 {n_mask}"
assert n_rand == 117, f"随机替换应 117，实际 {n_rand}"
assert n_keep == 120, f"原样保留应 120，实际 {n_keep}"
assert n_mask + n_rand + n_keep == n_sel
print("选中 %d / %d = %.6f" % (n_sel, int(M_all.sum()), n_sel / int(M_all.sum())))
print("  [MASK] %d | 随机替换 %d | 原样保留 %d" % (n_mask, n_rand, n_keep))
print()
print("为什么留 10%% 原样不动：否则 [MASK] 这个 token 只出现在预训练、微调时永远见不到，")
print("模型会学成「看到 [MASK] 才需要猜」这条捷径，两种阶段之间出现人为分布差。")

## 任务 6：两档语料预训练

In [ ]:
def pretrain(I, S, M, steps=500, seed=5):
    """无监督 MLM 预训练：每步采 32 条，只在被遮住的位置算交叉熵"""
    with torch.random.fork_rng():
        torch.manual_seed(seed)
        m = MiniBERT()
    opt = torch.optim.AdamW(m.parameters(), lr=1e-3)
    rgp = np.random.default_rng(77)
    m.train()
    for _ in range(steps):
        idx = rgp.integers(0, len(I), size=32)
        ids_b, lab_b = mlm_batch(I[idx], M[idx], rgp)
        # TODO(9)：前向 + 交叉熵；**必须 ignore_index=-100**，否则 padding 会被算进损失
        # 提示：F.cross_entropy(m.mlm_logits(...).reshape(-1, NV), lab_b.reshape(-1), ignore_index=-100)
        loss = ____
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        opt.step()
    m.eval()
    with torch.no_grad():
        ev_ids, ev_lab = mlm_batch(I, M, np.random.default_rng(999))
        lo = m.mlm_logits(ev_ids, S, M)
        msk = ev_lab != -100
        a1 = float((lo.argmax(-1)[msk] == ev_lab[msk]).float().mean())
        a5 = float((lo.topk(5, -1).indices[msk] == ev_lab[msk][:, None]).any(-1).float().mean())
        ent = float(-(F.softmax(lo, -1) * F.log_softmax(lo, -1)).sum(-1)[msk].mean())
    return m, float(loss), a1, a5, ent

In [ ]:
def gen_corpus(n, seed=7):
    """用 data/make_data.py 的同一套模板再生成 n 条**无标签**文本，做「预训练语料规模」对照"""
    spec = importlib.util.spec_from_file_location("mk", MAKE_DATA)
    mk = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mk)
    r9 = np.random.default_rng(seed)
    labels = list(mk.DEFECTS)
    out = []
    for i in range(n):
        lab = labels[i % len(labels)]
        rec = {
            "台区编号": str(r9.choice(mk.STATIONS)),
            "设备类型": str(r9.choice(mk.DEVICES)),
            "缺陷类型": lab,
            "缺陷等级": str(r9.choice(mk.LEVELS)),
            "_kw": str(r9.choice(mk.DEFECTS[lab])),
            "_act": str(r9.choice(mk.ACTIONS)),
            "_tpl": str(r9.choice(mk.TEMPLATES)),
        }
        out.append(mk.add_noise(mk.clean_text(rec), r9))
    return out


BIG = gen_corpus(20000)
BIG_TOK = [tok(clean(t)) for t in BIG]
I_big, S_big, M_big = featurize(BIG_TOK)
assert len(BIG) == 20000 and len(set(BIG)) == 19855, "去重后 19855 条"
assert tuple(I_big.shape) == (20000, 24) and int(M_big.sum()) == 333860
print("大语料 %d 条 | 去重 %d | 平均字符 %.4f | 平均 token %.4f"
      % (len(BIG), len(set(BIG)), np.mean([len(t) for t in BIG]),
         np.mean([len(t) for t in BIG_TOK])))
print("大语料张量 %s | 有效 token %d" % (tuple(I_big.shape), int(M_big.sum())))

BY_CLS = {c: [i for i in kept if df["缺陷类型"].iloc[i] == c] for c in LABELS5}


def split(FS, seed=0):
    """每类取 FS 条做 few-shot 训练，其余全部做测试"""
    rs = np.random.default_rng(seed)
    tr, te = [], []
    for c in LABELS5:
        pool = BY_CLS[c].copy()
        rs.shuffle(pool)
        tr += list(pool[:FS])
        te += list(pool[FS:])
    return tr, te


FINETUNE_SPLIT = split(20)
print("few-shot 划分（每类 20）：训练 %d / 测试 %d" % (len(FINETUNE_SPLIT[0]), len(FINETUNE_SPLIT[1])))

In [ ]:
# TODO(10)：两档语料各跑一次 MLM 预训练（只改语料池，步数/模型/超参完全一致）
# 提示：小语料 I_all(570 条)、大语料 I_big(20000 条)，都 500 步
m_small, l_small, a1_small, a5_small, e_small = ____
m_big, l_big, a1_big, a5_big, e_big = ____

SD_SMALL = copy.deepcopy(m_small.state_dict())
SD_BIG = copy.deepcopy(m_big.state_dict())

# ---- 验收 ----
assert abs(l_small - 0.465618) < 1e-5, f"小语料 MLM loss 应 0.465618，实际 {l_small}"
assert abs(a1_small - 0.863707) < 1e-5 and abs(a5_small - 0.943146) < 1e-5
assert abs(l_big - 0.487685) < 1e-5 and abs(a1_big - 0.827988) < 1e-5
assert a1_small > 10 / NV, "远高于随机基线"
print("小语料 570  条 | loss %.6f | 掩码位 top-1 %.6f | top-5 %.6f | 预测熵 %.6f"
      % (l_small, a1_small, a5_small, e_small))
print("大语料 20000 条 | loss %.6f | 掩码位 top-1 %.6f | top-5 %.6f | 预测熵 %.6f"
      % (l_big, a1_big, a5_big, e_big))
print("随机基线 top-1 = 1/%d = %.6f | top-5 = %.6f" % (NV, 1 / NV, 5 / NV))
print()
print("注意：大语料的 MLM 指标**更差**（top-1 %.6f < %.6f），因为池子更大、任务更难。" % (a1_big, a1_small))
print("但下一节会看到它的下游微调**更好** —— 预训练指标不等于表征质量。")

## 任务 7：预训练规模 × 下游数据量

In [ ]:
def fresh(sd, pool="mean"):
    """把预训练权重装进分类模型；pool 选 cls 或 mean"""
    m = mb(seed=9, pool=pool)
    m.load_state_dict(sd)
    return m


def fit(model, FS, steps=400, lr=1e-3, bs=32, seed=3):
    tr, te = split(FS)
    I_f, S_f, M_f = featurize([tokens[i] for i in tr])
    Y_f = torch.as_tensor([LID[df["缺陷类型"].iloc[i]] for i in tr])
    I_e, S_e, M_e = featurize([tokens[i] for i in te])
    Y_e = torch.as_tensor([LID[df["缺陷类型"].iloc[i]] for i in te])
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr)
    rg3 = np.random.default_rng(seed)
    model.train()
    for _ in range(steps):
        idx = rg3.integers(0, len(Y_f), size=min(bs, len(Y_f)))
        # TODO(11)：前向 + 分类交叉熵
        # 提示：F.cross_entropy(model(I_f[idx], S_f[idx], M_f[idx])[0], Y_f[idx])
        loss = ____
        opt.zero_grad()
        loss.backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        acc = float((model(I_e, S_e, M_e)[0].argmax(1) == Y_e).float().mean())
    return acc, len(tr), len(te)

In [ ]:
# TODO(12)：每类 40 条下，比三种「底座来源」
# 提示：大/小语料用 fresh(SD_XXX) 装权重；随机初始化用 mb(seed=9)
r_sm = ____
r_bg = ____
r_rd = ____

# ---- 验收 ----
assert abs(r_sm[0] - 0.694595) < 1e-5, f"小语料预训练应 0.694595，实际 {r_sm[0]}"
assert abs(r_bg[0] - 0.764865) < 1e-5, f"大语料预训练应 0.764865，实际 {r_bg[0]}"
assert abs(r_rd[0] - 0.910811) < 1e-5, f"随机初始化应 0.910811，实际 {r_rd[0]}"
assert r_bg[0] > r_sm[0], "扩大预训练语料确实提升了预训练权重的下游表现"
assert r_rd[0] > r_bg[0], "但在这个规模上，随机初始化仍然赢"
print("每类 40 条（训练 %d / 测试 %d）：" % (r_sm[1], r_sm[2]))
print("  小语料预训练 %.6f | 大语料预训练 %.6f | 随机初始化 %.6f" % (r_sm[0], r_bg[0], r_rd[0]))

In [ ]:
grid = {}
for FS in (5, 10, 20, 40):
    # TODO(13)：记下三列：小语料预训练 / 大语料预训练 / 随机初始化
    # 提示：a_small = fit(fresh(SD_SMALL), FS)[0]；依次类推
    a_small = ____
    a_big = ____
    a_rand = ____
    grid[FS] = ____

# ---- 验收 ----
assert abs(grid[5][0] - 0.249541) < 1e-5 and abs(grid[5][2] - 0.249541) < 1e-5
assert abs(grid[10][2] - 0.350000) < 1e-5
assert abs(grid[20][1] - 0.389362) < 1e-5 and abs(grid[20][2] - 0.591489) < 1e-5
assert abs(grid[40][1] - 0.764865) < 1e-5 and abs(grid[40][2] - 0.910811) < 1e-5
assert grid[5][0] == grid[5][2], "5 条/类时小语料预训练与随机初始化打平（都是 0.249541）"
assert sum(1 for k in grid if grid[k][2] > max(grid[k][0], grid[k][1])) == 3,     "随机初始化 3 胜 1 平 0 负"
assert all(grid[k][2] >= max(grid[k][0], grid[k][1]) for k in grid), "随机初始化 4 项均不劣"
assert grid[20][1] > grid[20][0] and grid[40][1] > grid[40][0], "样本够多时大语料优势才显现"
assert grid[5][1] < grid[5][0] and grid[10][1] < grid[10][0], "样本极少时大语料反而略差"
print("每类样本 | 小语料预训练 | 大语料预训练 | 随机初始化 | 胜者")
for FS in (5, 10, 20, 40):
    a, b, c = grid[FS]
    win = "随机初始化" if c > max(a, b) else ("打平" if c == max(a, b) else "预训练")
    print("    %-3d  |   %.6f   |   %.6f   |   %.6f   | %s" % (FS, a, b, c, win))
print()
print("两条反直觉但可复现的结论：")
print("  ① 5 条/类（训练集仅 25 条）时小语料预训练与随机初始化**完全打平**（都是 0.249541）——")
print("     此时决定上限的是「25 条数据」，不是底座好坏，预训练权重体现不出差异。")
print("  ② 大语料预训练在 5 / 10 条时反而**略差**（0.238532 / 0.267308），到 20 / 40 条才反超。")
print("     语料扩到 20000 条学到的是更细的词法统计，下游样本太少时来不及把它「翻译」成分类边界。")

## 任务 8：池化对比

In [ ]:
# TODO(14)：只换池化方式，其余（权重 / 划分 / 步数 / lr）完全一致
# 提示：fresh(SD_BIG, "cls") 与 fresh(SD_BIG, "mean")
acc_cls = ____
acc_mean = ____

# ---- 验收 ----
assert abs(acc_cls - 0.312766) < 1e-5, f"cls 池化应 0.312766，实际 {acc_cls}"
assert abs(acc_mean - 0.389362) < 1e-5, f"mean 池化应 0.389362，实际 {acc_mean}"
assert acc_mean > acc_cls
print("[CLS] 池化 %.6f | mean 池化 %.6f | 差 %+.6f" % (acc_cls, acc_mean, acc_mean - acc_cls))
print()
print("为什么这里 mean 更好：")
print("  MLM 逐位置监督过每个 token（除被 mask 的），所以「逐位置表征」是被训练过的；")
print("  而 [CLS] 只是句首占位符，MLM 目标**没有直接监督它**，它的表征要靠下游微调自己学。")
print("  100 条 few-shot 学不出来。真 BERT 的 [CLS] 好用，是因为有 NSP + 上亿句子。")

## 任务 9：手写 LoRA

In [ ]:
class LoRALinear(nn.Module):
    """h = W0·x + (alpha/r)·B·A·x —— 原权重冻结，只训 A、B 两个低秩矩阵"""

    def __init__(self, base, r, alpha):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        self.r = r
        self.scale = alpha / r
        # TODO(15)：定义 A(r, in) 与 B(out, r) 两个可训练参数；**B 必须全零**
        # 提示：B 全零 -> 训练开始时 BA = 0，模型行为与预训练权重逐元素一致
        self.A = ____
        self.B = ____
        with torch.random.fork_rng():
            torch.manual_seed(1234)
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    def forward(self, x):
        # TODO(16)：原层输出 + 缩放后的低秩增量
        # 提示：self.base(x) + self.scale * (x @ self.A.T @ self.B.T)
        ____


def to_lora(model, r=4, alpha=8.0, targets=("q", "v")):
    """把 attn.q / attn.v 换成 LoRALinear（LoRA 原论文就是挂在 Q/V 上）"""
    for name, mod in model.named_modules():
        if isinstance(mod, nn.Linear) and name.split(".")[-1] in targets:
            parent = model
            parts = name.split(".")
            for q in parts[:-1]:
                # ModuleList 的数字下标不支持 getattr，要用 [int(q)]
                parent = parent[int(q)] if q.isdigit() else getattr(parent, q)
            setattr(parent, parts[-1], LoRALinear(mod, r, alpha))
    return model

In [ ]:
lin = nn.Linear(256, 256)
z = torch.randn(3, 256)
rows = []
for r, exp in ((4, 2048), (8, 4096), (16, 8192)):
    ll = LoRALinear(lin, r=r, alpha=8.0)
    n_lora = sum(p.numel() for p in [ll.A, ll.B])
    # TODO(17)：算出「初始时输出与原层是否一致」以及「写入随机 A/B 后的最大差」
    # 提示：初始 B=0 -> BA=0；写入后用 (ll(z) - lin(z)).abs().max()
    same = ____
    n_b = ____
    assert n_lora == exp == r * (256 + 256), f"r={r} 参数量应 {exp}"
    assert same and n_b == ll.B.numel(), "B 全零时增量恒为 0"
    rows.append((r, n_lora, n_b, ll.B.numel()))

assert [x[1] for x in rows] == [2048, 4096, 8192]
assert all(x[2] == x[3] for x in rows), "B 的零元素个数 == B 的元素总数"
print("LoRA 参数量 = r × (in + out)：r=4 -> 2048 | r=8 -> 4096 | r=16 -> 8192")
print("原层参数 256×256+256 = %d，r=4 时 LoRA 只占 %.4f%%"
      % (256 * 256 + 256, 100 * 2048 / (256 * 256 + 256)))

ll = LoRALinear(lin, r=8, alpha=8.0)
with torch.no_grad():
    ll.B.normal_(0, 0.05)
    ll.A.normal_(0, 0.05)
d_after = float((ll(z) - lin(z)).abs().max())
assert d_after > 0.01, "B 填上随机值后必然改变输出"
W_merged = lin.weight + ll.scale * (ll.B @ ll.A)
d_merge = float((z @ W_merged.T + lin.bias - ll(z)).abs().max())
assert d_merge < 1e-5, f"合并后应等价，实际差 {d_merge}"
print("A/B 写随机值后输出最大差 %.6f（B 非零才开始起作用）" % d_after)
print("合并校验 W + s·B@A vs LoRA 前向 最大差 %.3e" % d_merge)
print("-> 推理时可把增量合并进原权重，部署零额外延迟（Adapter 做不到，这是 LoRA 更流行的关键）")

## 任务 10：LoRA 参数量与三策略

In [ ]:
# TODO(18)：算每个 r 下 LoRA 的可训参数与占比
# 提示：挂载点是每层 attn.q/attn.v 共 NL×2 处，每处 A(r×D)+B(D×r) = 2·r·D 个参数
# 提示：用字典推导一次算完：{r: NL * 2 * 2 * r * D for r in (4, 8, 16)}
LORA_N = ____
n_lora4 = ____

print("挂载点：每层 attn.q + attn.v，共 %d 层 × 2 个 = %d 个 Linear" % (NL, NL * 2))
for _r, _n in LORA_N.items():
    print("r=%-3d 可训参数 %5d | 占 Encoder %d 的 %.4f%% | 占全模型 %d 的 %.4f%%"
          % (_r, _n, enc_n, 100 * _n / enc_n, tot, 100 * _n / tot))

# ---- 验收 ----
assert n_lora4 == 2048 and LORA_N[16] == 8192
assert abs(100 * n_lora4 / tot - 1.794712) < 1e-5, "r=4 占全模型 1.7947%"
assert abs(100 * LORA_N[16] / tot - 7.178849) < 1e-5, "r=16 占 7.1788%"
assert LORA_N[8] == 4096
print("手算校验：r=4 时 2层 × 2个 × 2矩阵 × 4 × 64 = %d" % n_lora4)

In [ ]:
def run_strategy(tag, strategy, steps=400, bs=32, lr=1e-3, r=4):
    m = fresh(SD_BIG)
    if strategy == "head":
        for n, p in m.named_parameters():
            # TODO(19)：只训分类头：把非 head 的参数全部冻结
            # 提示：p.requires_grad = n.startswith("head.")
            p.requires_grad = ____
    elif strategy == "lora":
        m = to_lora(m, r=r)
        for n, p in m.named_parameters():
            # TODO(20)：只训 LoRA 的 A/B 与分类头
            # 提示：n.endswith(".A") or n.endswith(".B") or n.startswith("head.")
            p.requires_grad = ____
    params = [p for p in m.parameters() if p.requires_grad]
    ntr = sum(p.numel() for p in params)
    opt = torch.optim.AdamW(params, lr=lr)
    rg4 = np.random.default_rng(3)
    tr, te = FINETUNE_SPLIT
    I_f, S_f, M_f = featurize([tokens[i] for i in tr])
    Y_f = torch.as_tensor([LID[df["缺陷类型"].iloc[i]] for i in tr])
    I_e, S_e, M_e = featurize([tokens[i] for i in te])
    Y_e = torch.as_tensor([LID[df["缺陷类型"].iloc[i]] for i in te])
    m.train()
    for _ in range(steps):
        idx = rg4.integers(0, len(Y_f), size=bs)
        # TODO(21)：训练一步
        # 提示：前向取 [0]（分类 logits），再 F.cross_entropy
        loss = ____
        opt.zero_grad()
        loss.backward()
        opt.step()
    m.eval()
    with torch.no_grad():
        acc = float((m(I_e, S_e, M_e)[0].argmax(1) == Y_e).float().mean())
    return acc, ntr

In [ ]:
acc_full, n_full = run_strategy("全量微调", "full")
acc_head, n_head = run_strategy("只训分类头", "head")
acc_l4, n_l4 = run_strategy("LoRA r=4", "lora", r=4)
acc_l16, n_l16 = run_strategy("LoRA r=16", "lora", r=16)

# ---- 验收 ----
assert (n_full, n_head, n_l4, n_l16) == (114113, 325, 2373, 8517), \
    f"可训参数不符：{(n_full, n_head, n_l4, n_l16)}"
assert abs(acc_full - 0.389362) < 1e-5, f"全量 {acc_full}"
assert abs(acc_head - 0.236170) < 1e-5, f"head {acc_head}"
assert abs(acc_l4 - 0.270213) < 1e-5, f"lora4 {acc_l4}"
assert abs(acc_l16 - 0.257447) < 1e-5, f"lora16 {acc_l16}"
assert acc_l4 > acc_head, "LoRA 明显优于只训分类头"
assert acc_l4 / acc_full > 0.69, "LoRA 用 2.08% 参数拿到全量的近 70%"
assert acc_l16 < acc_l4, "r=16 反而更差（数据少，秩大就过拟合）"
print("%-14s %8s %10s %10s" % ("策略", "可训参数", "占比", "测试准确"))
for tag, a, n in (("全量微调", acc_full, n_full), ("只训分类头", acc_head, n_head),
                  ("LoRA r=4", acc_l4, n_l4), ("LoRA r=16", acc_l16, n_l16)):
    print("%-14s %8d %9.4f%% %10.6f" % (tag, n, 100 * n / tot, a))

## 任务 11：HF BERT 参数量恒等式

用 `transformers` 建同配置模型（**只建 config，不下载任何权重**），
填出与手写 MiniBERT 的参数量恒等式。

⚠️ MPS 上必须把 `attention_probs_dropout_prob` 设为 `0.0`，
否则 `scaled_dot_product_attention` 会直接 `NotImplementedError`。

In [ ]:
# 微调用的划分与张量：口径与 §7.9 的 run_strategy 内部完全一致
FT_TR, FT_TE = FINETUNE_SPLIT
I_f, S_f, M_f = featurize([tokens[i] for i in FT_TR])
Y_f = torch.as_tensor([LID[df["缺陷类型"].iloc[i]] for i in FT_TR])
I_e, S_e, M_e = featurize([tokens[i] for i in FT_TE])
Y_e = torch.as_tensor([LID[df["缺陷类型"].iloc[i]] for i in FT_TE])
print("few-shot 张量：训练 %s / 测试 %s（每类 20 条）"
      % (tuple(I_f.shape), tuple(I_e.shape)))

# ---- 验收 ----
assert tuple(I_f.shape) == (100, 24) and tuple(I_e.shape) == (470, 24)
assert len(Y_f) == 100 and len(Y_e) == 470, "每类 20 条 × 5 类 = 100 条训练"

In [ ]:
cfg_hf = BertConfig(vocab_size=NV, hidden_size=D, num_hidden_layers=NL,
                    num_attention_heads=H, intermediate_size=FF,
                    max_position_embeddings=ML, num_labels=5,
                    hidden_dropout_prob=0.0, attention_probs_dropout_prob=0.0)


def build_hf(seed=9):
    """同配置的 HF BERT —— 仍然是 from_config 随机初始化，不下载任何预训练权重"""
    with torch.random.fork_rng():
        torch.manual_seed(seed)
        return BertForSequenceClassification(cfg_hf)


hf_ref = build_hf()
n_emb = sum(p.numel() for n_, p in hf_ref.named_parameters() if "embeddings" in n_)
n_enc = sum(p.numel() for n_, p in hf_ref.named_parameters() if ".layer." in n_)
n_pool = sum(p.numel() for n_, p in hf_ref.named_parameters() if "pooler" in n_)
n_head = sum(p.numel() for n_, p in hf_ref.named_parameters() if "classifier" in n_)

# TODO(22)：HF 版与手写 MiniBERT 的参数恒等式：MiniBERT 去掉 MLM 头，再加上 pooler
# 提示：手写 MiniBERT 总参 114113，其中 MLM 头 24828；HF 多出 pooler
N_MLM = ____
hf_expect = ____

print("HF BertForSequenceClassification 总参数:", hf_ref.num_parameters())
print("  拆解: 嵌入 %d | encoder %d | pooler %d | 分类头 %d"
      % (n_emb, n_enc, n_pool, n_head))
print("恒等式: 114113 - %d(MLM 头) + %d(pooler) = %d" % (N_MLM, n_pool, hf_expect))

# ---- 验收 ----
assert hf_ref.num_parameters() == 93445
assert (n_emb, n_enc, n_pool, n_head) == (22016, 66944, 4160, 325)
assert n_emb + n_enc + n_pool + n_head == 93445
assert hf_expect == 93445

## 任务 12：把手写 LoRA 挂到 HF 模型上

`to_lora()` 是你自己在任务 9 里写的，**一行都不用改**，只需要换 `targets`。

⚠️ HF 的层名是 `query` / `value`，**不是** `q` / `v`。

⚠️ 替换完还要**再冻结一遍**其余参数 —— `to_lora` 只冻结了被替换的那个 `base` 层。

In [ ]:
hf = build_hf()

# TODO(23)：用 ch07 自己写的 to_lora() 把 HF 的 query / value 投影换成 LoRALinear
# 提示：HF 的层名是 query / value，**不是** q / v —— targets 必须显式传
# 提示：to_lora(hf, r=4, targets=("query", "value"))
____

hit = [n_ for n_, mod in hf.named_modules() if isinstance(mod, LoRALinear)]
n_all = sum(p.numel() for p in hf.parameters())

for n_, p in hf.named_parameters():
    # TODO(24)：只训 LoRA 的 A / B 与分类头，其余全部冻结
    # 提示：n_.endswith(".A") or n_.endswith(".B") or n_.startswith("classifier.")
    p.requires_grad = ____
n_grad = sum(p.numel() for p in hf.parameters() if p.requires_grad)

print("替换到的层（%d 个）:" % len(hit))
for n_ in hit:
    print("   ", n_)
print("总参数 %d | 可训参数 %d（%.6f%%）" % (n_all, n_grad, 100 * n_grad / n_all))

# ---- 验收 ----
assert len(hit) == 4
assert hit[0] == "bert.encoder.layer.0.attention.self.query"
assert n_all == 95493
assert n_grad == 2373
assert abs(100 * n_grad / n_all - 2.484999) < 1e-5

## 任务 13：四种策略在 HF BERT 上真跑

把任务 10 的四种策略原样搬到 HF 底座上。

⚠️ 拿到 0.959574 这个高分时先别下结论 —— **底座来源不同**，下一题才是公平对照。

In [ ]:
def hf_fit(strategy="full", r=4, steps=400, bs=32, lr=1e-3, seed=3):
    m = build_hf()
    if strategy == "head":
        for n_, p in m.named_parameters():
            p.requires_grad = n_.startswith("classifier.")
    elif strategy == "lora":
        to_lora(m, r=r, targets=("query", "value"))
        for n_, p in m.named_parameters():
            p.requires_grad = (n_.endswith(".A") or n_.endswith(".B")
                               or n_.startswith("classifier."))
    m = m.to(DEV_HF)
    params = [p for p in m.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr)
    rg = np.random.default_rng(seed)
    m.train()
    hist = []
    for _ in range(steps):
        idx = torch.as_tensor(rg.integers(0, len(Y_f), size=bs))
        out = m(input_ids=I_f[idx].to(DEV_HF), attention_mask=M_f[idx].to(DEV_HF),
                labels=Y_f[idx].to(DEV_HF))
        opt.zero_grad()
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        opt.step()
        hist.append(out.loss.item())
    m.eval()
    with torch.no_grad():
        lg = m(input_ids=I_e.to(DEV_HF), attention_mask=M_e.to(DEV_HF)).logits
    acc = float((lg.argmax(-1).cpu() == Y_e).float().mean())
    return acc, sum(p.numel() for p in params), float(np.mean(hist[-50:]))


# TODO(25)：四种策略各跑一次，结果存成 [(tag, (acc, 可训参数, 末段loss)), ...]
# 提示：列表推导式：[(tag, hf_fit(**kw)) for tag, kw in [...]]
HF_RES = ____

for tag, (acc, n, loss) in HF_RES:
    print("  %-12s 测试准确 %.6f | 可训参数 %7d | 末50步 loss %.6f" % (tag, acc, n, loss))
print("对照 ch07 手写 MiniBERT（同数据同步数）: 全量 0.389362 / head 0.236170 / LoRA r=4 0.270213")

# ---- 验收 ----
assert abs(HF_RES[0][1][0] - 0.959574) < 1e-6
assert abs(HF_RES[1][1][0] - 0.200000) < 1e-6
assert abs(HF_RES[2][1][0] - 0.348936) < 1e-6
assert abs(HF_RES[3][1][0] - 0.310638) < 1e-6
assert HF_RES[0][1][1] == 93445 and HF_RES[2][1][1] == 2373

## 任务 14：公平对照（本章最有价值的一题）

同一套手写 MiniBERT，一边装 MLM 预训练权重、一边纯随机初始化，
跑**完全相同**的四种策略。

⚠️ 结论会反直觉：**随机初始化 4 胜 0 负**。

In [ ]:
def mini_fit(strategy="full", r=4, steps=400, bs=32, lr=1e-3, seed=3):
    """同样的四种策略，但底座换成「全新随机初始化」的 MiniBERT"""
    m = mb(seed=9, pool="mean")
    if strategy == "head":
        for n_, p in m.named_parameters():
            p.requires_grad = n_.startswith("head.")
    elif strategy == "lora":
        m = to_lora(m, r=r)
        for n_, p in m.named_parameters():
            p.requires_grad = (n_.endswith(".A") or n_.endswith(".B")
                               or n_.startswith("head."))
    params = [p for p in m.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr)
    rg = np.random.default_rng(seed)
    m.train()
    for _ in range(steps):
        idx = rg.integers(0, len(Y_f), size=bs)
        loss = F.cross_entropy(m(I_f[idx], S_f[idx], M_f[idx])[0], Y_f[idx])
        opt.zero_grad()
        loss.backward()
        opt.step()
    m.eval()
    with torch.no_grad():
        acc = float((m(I_e, S_e, M_e)[0].argmax(1) == Y_e).float().mean())
    return acc, sum(p.numel() for p in params)


# ch07 §7.8 / §7.11 已交付的真值：从 MLM 预训练权重 SD_BIG 出发
PRETRAINED = {"全量微调": 0.389362, "只训分类头": 0.236170,
              "LoRA r=4": 0.270213, "LoRA r=16": 0.257447}

# TODO(26)：随机初始化跑同样四种策略，为「随机 vs 预训练」的胜负计分
# 提示：列表推导式：[(tag, mini_fit(**kw)[0]) for tag, kw in [...]]
RAND = ____

win = sum(1 for tag, a in RAND if a > PRETRAINED[tag])
print("%-12s %12s %12s %14s" % ("策略", "随机初始化", "预训练权重", "随机 - 预训练"))
for tag, a in RAND:
    print("%-12s %12.6f %12.6f %+14.6f" % (tag, a, PRETRAINED[tag], a - PRETRAINED[tag]))
print("随机初始化战绩: %d 胜 / %d 负（共 %d 项）"
      % (win, len(RAND) - win, len(RAND)))

# ---- 验收 ----
assert win == 4, "每类 20 条下，随机初始化在全部 4 种策略上都优于预训练权重"
assert abs(RAND[0][1] - 0.617021) < 1e-6
assert abs(RAND[2][1] - 0.412766) < 1e-6

## 任务 15：用 Trainer 真跑

补一个 `Dataset` + 一次 `Trainer.train()`，然后对比四种 lr 调度。

⚠️ `Trainer` 默认 `lr_scheduler_type="linear"`，在 400 步这种短训练上会把 lr
提前衰减到 0 —— **默认值直接让准确率掉 0.0596**。

⚠️ transformers 5.17 的 `TrainingArguments` **没有 `warmup_ratio`**，只有 `warmup_steps`。

In [ ]:
from torch.utils.data import Dataset


class TicketDS(Dataset):
    """给 Trainer 用的最小数据集：返回 dict，键名与 HF forward 的参数名一致"""

    def __init__(self, I, M, Y):
        self.I, self.M, self.Y = I, M, Y

    def __len__(self):
        return len(self.Y)

    def __getitem__(self, i):
        return {"input_ids": self.I[i], "attention_mask": self.M[i], "labels": self.Y[i]}


def trainer_run(sched, steps=400, lr=1e-3):
    m = build_hf()
    to_lora(m, r=4, targets=("query", "value"))
    for n_, p in m.named_parameters():
        p.requires_grad = (n_.endswith(".A") or n_.endswith(".B")
                           or n_.startswith("classifier."))
    args = TrainingArguments(output_dir="/tmp/ch07_out", per_device_train_batch_size=32,
                             max_steps=steps, learning_rate=lr, logging_steps=steps // 4,
                             report_to=[], save_strategy="no", eval_strategy="no",
                             disable_tqdm=True, seed=3, dataloader_num_workers=0,
                             warmup_steps=0, lr_scheduler_type=sched)
    tr = Trainer(model=m, args=args, train_dataset=TicketDS(I_f, M_f, Y_f))
    out = tr.train()
    pred = tr.predict(TicketDS(I_e, M_e, Y_e))
    acc = float((pred.predictions.argmax(-1) == Y_e.numpy()).mean())
    return acc, out.metrics["train_loss"]


print("Trainer 的默认 lr 调度:",
      TrainingArguments(output_dir="/tmp/probe", report_to=[], disable_tqdm=True).lr_scheduler_type)

# TODO(27)：四种 lr 调度各跑一次，看「默认值」到底有多影响结果
# 提示：列表推导式：[(s, trainer_run(s)) for s in (...)]
TR_RES = ____

for s, (acc, loss) in TR_RES:
    print("  %-22s 测试准确 %.6f | train_loss %.6f" % (s, acc, loss))
print("对照手写 loop（恒定 lr，400 步）: 测试准确 0.348936 | 末50步 loss 1.043668")

# ---- 验收 ----
assert abs(TR_RES[0][1][0] - 0.257447) < 1e-6, "默认 linear 调度"
assert abs(TR_RES[1][1][0] - 0.317021) < 1e-6, "constant"
assert TR_RES[1][1][0] == TR_RES[2][1][0], "warmup_steps=0 时 constant 与 constant_with_warmup 等价"
assert abs(TR_RES[3][1][0] - 0.255319) < 1e-6, "cosine"

## 自查清单

- [ ] WordPiece 词表 **316** 词、续接片段 **151** 个
- [ ] `wp_encode("请核实") == ["请", "##核", "##实"]`
- [ ] `wp_encode("铁芯") == ["[UNK]"]`
- [ ] 单句有效长度 4、句对 8、`seg = [0,0,0,0,1,1,1,1]`
- [ ] 截断 `maxlen=3` 得到 `[CLS] 变压器 [SEP]`
- [ ] 参数量 22016 + 66944 + 325 + 24828 = **114113**
- [ ] 掩码选中 **1307**（13.48%）：[MASK] **1070** / 随机 **117** / 保留 **120**
- [ ] 小语料 MLM top-1 **0.863707**；大语料 **0.827988**
- [ ] 数据量矩阵中**随机初始化 3 胜 1 平 0 负**，5 条/类时三家打平 0.249541
- [ ] `[CLS]` **0.312766** < mean **0.389362**
- [ ] LoRA r=4 可训参数 **2373**（**2.0795%**），效果 **0.270213**
- [ ] `B` 全零时 LoRA 输出与原层逐元素一致；合并差 **1.25e-06**
- [ ] 三策略可训参数 = **114113 / 325 / 2373 / 8517**
- [ ] HF BERT 总参 **93445** = 114113 − 24828(MLM 头) + **4160**(pooler)
- [ ] `to_lora(..., targets=("query","value"))` 命中 **4** 层；总参 95493、可训 **2373**（**2.485%**）
- [ ] HF 四策略：全量 **0.959574** / head **0.200000** / LoRA r=4 **0.348936** / r=16 **0.310638**
- [ ] **随机初始化 4 胜 0 负**（0.617021 vs 0.389362 等）—— 核心负结果的二次复现
- [ ] Trainer 默认 **linear** 调度 **0.257447** < `constant` **0.317021**；`warmup_steps=0` 时两者等价

**答不上来时问自己**

1. `token_type_ids` 在单句任务里为什么全 0？什么时候才有用？
2. 为什么 `##` 加在续接片段上，而不是给词首加前缀？
3. MLM 为什么留 10% 原样不动？
4. 为什么大语料的 MLM 指标更差、下游却更好？
5. LoRA 的 `B` 不置零会怎样？
6. 为什么这里 LoRA r=16 反而比 r=4 差？
7. HF 版比手写 MiniBERT **少的 24828** 和**多出的 4160** 分别是什么？
8. HF 全量微调拿到 0.9596，为什么不能推出「真实库比手写强」？
9. `Trainer` 默认的 lr 调度是什么？为什么它在 400 步这种短训练上会坑人？
10. 如果预训练权重在这个规模下是负担，那它要**在什么条件下**才变成资产？